# FM study: development, nomination and reports

This notebook computes the study's **first labelled scores**. Run it only after Lam has approved the
input-lock SHA256 and recorded the public timestamp (amendments A2 A-8 and A3 S-5). It needs no GPU and
no Hugging Face token: it reads the lock and the features that `colab_fm.ipynb` saved to
`octfsl/fm_v1/final/<attempt>/`.

**Step-by-step guide:** `docs/research/FM_FIRST_RUN_GUIDE.md`, Part 4.

| Step | What it does | What proves it worked |
|---|---|---|
| 1 | Checks the first cell | Every field filled; the SHA256 has 64 characters |
| 2 | Clones the exact code and builds the same Python environment | Commit matches; `pip check` passes |
| 3 | Copies the lock and features from Drive, then disconnects Drive | Every file matches the saved receipt and Lam's SHA256 |
| 4 | Restores the lock and unpacks the features | Lock restored; same Python, NumPy and SciPy as at the lock |
| 5 | Development: K-shot episodes on the five folds, every model | One `development` folder per model |
| 6 | Nominates one model, saves the decision to Drive, **then stops** | The nomination is printed |
| 7 | Reports: legacy 22 images, nine-way audit, external images | One `reporting` and one `external` folder per model |
| 8 | Summary tables and the geometry table | Files listed |
| 9 | Saves everything to `octfsl/fm_v1/final/<attempt>/evaluation/` | The Drive copy's SHA256 matches |

Step 6 stops on purpose. Lam reads the nomination, sets `REPORT_AFTER_NOMINATION = True` in the first
cell, and the remaining steps are then run. The nominee can never change after that: the code seals it.
Nothing here may be edited to change a result.

In [ ]:
# Fill in this cell before Run all (Step 1 stops until it is complete)
ATTEMPT = ""            # printed at the end of colab_fm.ipynb, for example final-20261114T030000Z
LOCK_SHA256 = ""        # the input-lock SHA256 Lam approved (64 characters)
LOCK_APPROVAL = ""      # Lam's approval: name, date and words
TIMESTAMP_RECORD = ""   # where and when the hashes were deposited, or emailed to the mentor and team
RUN_BY = ""             # your name
# Lam sets this to True after reading the nomination that Step 6 prints.
REPORT_AFTER_NOMINATION = False
# Leave empty. Only Lam fills it, for a dated and disclosed reduction (protocol section 9).
OMIT_LOGISTIC_REASON = ""

In [ ]:
# Settings and helpers
import datetime, hashlib, json, os, pathlib, shlex, shutil, subprocess, sys, zipfile

COMMIT = "41782db6cf3de39155d21f77c0b88971e6317ae2"  # the locked code; change only on purpose
REPO = "https://github.com/Malnoad/Few-shot-rare-diseases"
ROOT, ENV = "/content/Few-shot", "/content/fm-env"
PY = f"{ENV}/bin/python"
EVAL = "/content/fm_eval"  # outside the code checkout, as the scripts require
ARCHIVES, LOCK, FEATURES, EXTERNAL = f"{EVAL}/archives", f"{EVAL}/lock", f"{EVAL}/features", f"{EVAL}/external"
RUNS, TABLES, NOMINATION = f"{EVAL}/runs", f"{EVAL}/reports", f"{EVAL}/reports/nomination"
OUT = "/content/drive/MyDrive/octfsl/fm_v1"  # Lam's output folder, through the runner's octfsl shortcut
ARMS = ("RETFound", "DINOv2", "MIRAGE", "Inception")


def sh(command):
    """Run a shell command, streaming its output; any failure stops the notebook."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"FAILED: {command}")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_to_drive(folder, paths, replace=False):
    """Copy files into octfsl/fm_v1/<folder>, verify every copy, then disconnect Drive.
    A file already there with the same bytes is kept; different bytes are never overwritten,
    except a receipt saved with replace=True."""
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    try:
        target = pathlib.Path(OUT, folder)
        target.mkdir(parents=True, exist_ok=True)
        for path in map(pathlib.Path, paths):
            digest, copy = sha256(path), target / path.name
            if copy.exists() and not replace:
                assert sha256(copy) == digest, f"{copy} already exists with different bytes; nothing was overwritten"
            else:  # copy under a temporary name first, so an interrupted copy never blocks a retry
                pending = target / (path.name + ".part")
                shutil.copyfile(path, pending)
                assert sha256(pending) == digest, (f"The Drive copy of {path.name} differs: run this step again; if it fails "
                                                   f"again, download {path} from the Files panel and send it to Lam.")
                os.replace(pending, copy)
            print(f"saved {path.name}  {digest}")
    finally:
        drive.flush_and_unmount()

In [ ]:
# Step 1. Check the first cell
first_cell = dict(ATTEMPT=ATTEMPT, LOCK_SHA256=LOCK_SHA256, LOCK_APPROVAL=LOCK_APPROVAL,
                  TIMESTAMP_RECORD=TIMESTAMP_RECORD, RUN_BY=RUN_BY)
blank = [name for name, value in first_cell.items() if not value.strip()]
assert not blank, f"Stopped on purpose: fill in {blank} in the first cell, run that cell, then Run all again."
template = [name for name, value in first_cell.items() if "[" in value or "<" in value]
assert not template, f"Stopped on purpose: {template} still contain a [ or < placeholder; replace it with the real text."
LOCK_SHA256 = LOCK_SHA256.strip().lower()
assert len(LOCK_SHA256) == 64 and set(LOCK_SHA256) <= set("0123456789abcdef"), "LOCK_SHA256 must be the 64-character SHA256 Lam approved."
assert ATTEMPT.strip().startswith("final-"), "ATTEMPT must be the name printed by colab_fm.ipynb, starting with final-"
ATTEMPT = ATTEMPT.strip()
assert isinstance(REPORT_AFTER_NOMINATION, bool), "REPORT_AFTER_NOMINATION must be True or False."
print("First cell complete. Attempt:", ATTEMPT)

In [ ]:
# Step 2. Get the exact code and build the same Python environment as at the lock
if not os.path.isdir(ROOT):
    sh(f"git clone -q {REPO} {ROOT}")
sh(f"git -C {ROOT} checkout -q {COMMIT}")
head = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, head
print("code:", head)
if not os.path.exists(PY):
    sh(f"python3 -m venv --without-pip {ENV}")
sh(f"python3 -m pip --python {PY} install -q -r {ROOT}/requirements-fm.txt")
sh(f"python3 -m pip --python {PY} check")
sh(f"""{PY} -c 'import sys, numpy, scipy; print("python", sys.version.split()[0], "numpy", numpy.__version__, "scipy", scipy.__version__)'""")

In [ ]:
# Step 3. Copy the lock and features from Drive, check them, then disconnect Drive
from google.colab import drive

SOURCE = f"{OUT}/final/{ATTEMPT}"
os.makedirs(ARCHIVES, exist_ok=True)
if not os.path.exists(f"{ARCHIVES}/receipt.json"):
    drive.mount("/content/drive", force_remount=True)
    try:
        assert os.path.isfile(f"{SOURCE}/receipt.json"), f"{SOURCE}/receipt.json not found: check ATTEMPT and the octfsl shortcut."
        for name in sorted(os.listdir(SOURCE)):
            if os.path.isfile(f"{SOURCE}/{name}") and name != "receipt.json":
                shutil.copyfile(f"{SOURCE}/{name}", f"{ARCHIVES}/{name}")
        saved = f"{SOURCE}/evaluation/nomination_binding.json"  # present only if a nomination was already made
        if os.path.isfile(saved):
            shutil.copyfile(saved, f"{ARCHIVES}/nomination_binding.json")
        shutil.copyfile(f"{SOURCE}/receipt.json", f"{ARCHIVES}/receipt.json")
    finally:
        drive.flush_and_unmount()
RECEIPT = json.loads(pathlib.Path(ARCHIVES, "receipt.json").read_text(encoding="utf-8"))
assert RECEIPT["attempt"] == ATTEMPT and RECEIPT["complete"], "The saved receipt is for another attempt or is incomplete: tell Lam."
assert RECEIPT["code"] == COMMIT, f"The features were made with code {RECEIPT['code']}, not this notebook's {COMMIT}: tell Lam."
assert RECEIPT["lock_sha256"] == LOCK_SHA256, "LOCK_SHA256 in the first cell is not the lock of this attempt: check with Lam."
for name, entry in RECEIPT["files"].items():
    assert sha256(f"{ARCHIVES}/{name}") == entry["sha256"], f"{name} differs from the saved receipt: tell Lam."
assert sha256(f"{ARCHIVES}/input-lock.zip") == LOCK_SHA256
print(len(RECEIPT["files"]), "archives match the receipt; the lock is the one Lam approved.")

In [ ]:
# Step 4. Restore the lock and unpack the features
# restore is safe to repeat: it writes each file once and finishes with the receipt
sh(f"{PY} {ROOT}/fm_prepare.py restore --root {ROOT} --out {LOCK} --archive {ARCHIVES}/input-lock.zip --sha256 {LOCK_SHA256}")
if os.path.exists(f"{ARCHIVES}/nomination_binding.json") and not os.path.exists(f"{LOCK}/nomination_binding.json"):
    shutil.copyfile(f"{ARCHIVES}/nomination_binding.json", f"{LOCK}/nomination_binding.json")  # the original decision
    print("Restored the nomination already made for this lock; it will not be recomputed.")


def unpack(archive, folder):
    """Unpack beside the target and rename at the end, so a folder that exists is always complete."""
    if not os.path.isdir(folder):
        shutil.rmtree(folder + ".part", ignore_errors=True)
        with zipfile.ZipFile(archive) as zipped:
            zipped.extractall(folder + ".part")
        os.makedirs(os.path.dirname(folder), exist_ok=True)
        os.replace(folder + ".part", folder)


LOCKED = json.loads(pathlib.Path(LOCK, "input_lock.json").read_text(encoding="utf-8"))
AVAILABLE = [arm for arm in ARMS if LOCKED["arms"][arm]["status"] == "available"]
unpack(f"{ARCHIVES}/external-lock.zip", EXTERNAL)
for arm in AVAILABLE:
    unpack(f"{ARCHIVES}/features-{arm}.zip", f"{FEATURES}/{arm}")
    unpack(f"{ARCHIVES}/features-{arm}-external.zip", f"{FEATURES}/{arm}-external")
current = json.loads(subprocess.check_output([PY, "-c", "import json, fm_prepare; print(json.dumps(fm_prepare.math_environment()))"], cwd=ROOT, text=True))
assert current == LOCKED["evaluation_environment"], (
    f"This runtime has {current}, the lock has {LOCKED['evaluation_environment']}. Scores must use the locked versions: stop and tell Lam.")
APPROVALS = f"{EVAL}/approvals.json"
if not os.path.exists(APPROVALS):
    pathlib.Path(APPROVALS).write_text(json.dumps(dict(
        attempt=ATTEMPT, lock_sha256=LOCK_SHA256, lock_approval=LOCK_APPROVAL.strip(), timestamp_record=TIMESTAMP_RECORD.strip(),
        run_by=RUN_BY.strip(), code=COMMIT, recorded_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")),
        indent=2) + "\n", encoding="utf-8")
OMIT = f" --omit-logistic-reason {shlex.quote(OMIT_LOGISTIC_REASON.strip())}" if OMIT_LOGISTIC_REASON.strip() else ""
print("Lock restored. Available models:", AVAILABLE, " environment:", current)

In [ ]:
# Step 5. Development: paired K-shot episodes on the five folds, every available model (first labelled scores)
for arm in AVAILABLE:
    out = f"{RUNS}/{arm}/development"
    if not os.path.exists(f"{out}/complete.json"):
        sh(f"{PY} {ROOT}/fm_evaluate.py develop --lock {LOCK} --cache {FEATURES}/{arm} --arm {arm} --out {out}{OMIT}")
    print("development complete:", arm)

In [ ]:
# Step 6. Nominate one model from the development folds, save the decision to Drive, then stop for Lam
if not os.path.exists(f"{NOMINATION}/complete.json"):
    sh(f"{PY} {ROOT}/fm_evaluate.py nominate --lock {LOCK} --runs {RUNS} --out {NOMINATION}")
os.makedirs(f"{EVAL}/exports", exist_ok=True)
sh(f"{PY} {ROOT}/fm_prepare.py export --root {ROOT} --out {NOMINATION} --archive {EVAL}/exports/nomination.zip")
BINDING_SHA256 = sha256(f"{LOCK}/nomination_binding.json")
save_to_drive(f"final/{ATTEMPT}/evaluation", [f"{LOCK}/nomination_binding.json", f"{EVAL}/exports/nomination.zip",
                                               f"{EVAL}/exports/nomination.zip.json"])
print(pathlib.Path(NOMINATION, "nomination.json").read_text(encoding="utf-8"))
print("Nomination sealed. Binding SHA256:", BINDING_SHA256)
assert REPORT_AFTER_NOMINATION, ("Stopped on purpose. Send Lam the nomination printed above. When Lam has read it, set "
                                 "REPORT_AFTER_NOMINATION = True in the first cell, run that cell, then run Step 6 onwards.")

In [ ]:
# Step 7. Reports: legacy 22 images and nine-way audit, then external images, every available model
for arm in AVAILABLE:
    out = f"{RUNS}/{arm}/reporting"
    if not os.path.exists(f"{out}/complete.json"):
        sh(f"{PY} {ROOT}/fm_evaluate.py report --lock {LOCK} --cache {FEATURES}/{arm} --arm {arm} --out {out} "
           f"--nomination {NOMINATION}{OMIT}")
    out = f"{RUNS}/{arm}/external"
    if not os.path.exists(f"{out}/complete.json"):
        sh(f"{PY} {ROOT}/fm_external.py report --lock {LOCK} --cache {FEATURES}/{arm} --external-cache {FEATURES}/{arm}-external "
           f"--external {EXTERNAL} --arm {arm} --nomination {NOMINATION} --out {out}{OMIT}")
    print("reports complete:", arm)

In [ ]:
# Step 8. Summary tables and the geometry table
if not os.path.exists(f"{TABLES}/legacy/complete.json"):
    sh(f"{PY} {ROOT}/fm_evaluate.py summarize --lock {LOCK} --runs {RUNS} --nomination {NOMINATION} --out {TABLES}/legacy")
if not os.path.exists(f"{TABLES}/external/complete.json"):
    sh(f"{PY} {ROOT}/fm_external.py summarize --lock {LOCK} --external {EXTERNAL} --runs {RUNS} --nomination {NOMINATION} "
       f"--out {TABLES}/external")
for arm in AVAILABLE:
    out = f"{TABLES}/geometry/{arm}"
    if not os.path.exists(f"{out}/complete.json"):
        sh(f"{PY} {ROOT}/fm_evaluate.py geometry --lock {LOCK} --cache {FEATURES}/{arm} --arm {arm} --out {out} "
           f"--external {EXTERNAL} --external-cache {FEATURES}/{arm}-external")
for path in sorted(pathlib.Path(TABLES).rglob("*.csv")):
    print(path.relative_to(TABLES).as_posix())

In [ ]:
# Step 9. Save runs and tables to octfsl/fm_v1/final/<attempt>/evaluation/, then disconnect Drive
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
results = f"{EVAL}/exports/results-{stamp}.zip"
with zipfile.ZipFile(results, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (RUNS, TABLES):
        for path in sorted(pathlib.Path(folder).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(EVAL).as_posix())
    archive.write(APPROVALS, "approvals.json")
receipt = dict(attempt=ATTEMPT, run_by=RUN_BY.strip(), code=COMMIT, lock_sha256=LOCK_SHA256, binding_sha256=BINDING_SHA256,
               zip=os.path.basename(results), zip_sha256=sha256(results), zip_bytes=os.path.getsize(results), saved_utc=stamp)
receipt_path = f"{EVAL}/exports/results-{stamp}.json"
pathlib.Path(receipt_path).write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
save_to_drive(f"final/{ATTEMPT}/evaluation", [results, receipt_path])  # new names each run, so nothing is replaced
print(f"All done. Tell Lam: results saved as {os.path.basename(results)}, SHA256 {receipt['zip_sha256']}")